In [1]:
import numpy as np
%run liga_f.ipynb
import pandas as pd

# Read the Liga F all fixtures CSV file
wsl_df = pd.read_csv('data/wsl/wsl_all_fixtures_2025-10-30.csv')
wsl_df['home_score'] = wsl_df['home_score'].fillna(0.0)
wsl_df['away_score'] = wsl_df['away_score'].fillna(0.0)

Original dataset shape: (960, 20)

Available seasons:
season
2022-2023    240
2023-2024    240
2024-2025    240
2025-2026    240
Name: count, dtype: int64

Filtered dataset for 2022-2023 season shape: (240, 20)
Number of matches: 240
Home Win Coefficients: [0.25 0.5  0.75 1.   1.25 1.5  1.75]
Away Win Coefficients: [0.25 0.5  0.75 1.   1.25 1.5  1.75]
Margin Coefficients: [0.  0.5 1. ]
Date Coefficients: [0.  0.5 1. ]
New best found: Home Coeff=0.25, Away Coeff=0.25, Margin Coeff=0.00, Date Coeff=0.00, Error=7.7460, Mean%Diff=0.1389
predicted ranked teams:  ['Barcelona', 'Real Madrid', 'Levante', 'Atlético Madrid', 'Madrid CFF', 'Real Sociedad', 'Valencia', 'Sporting Huelva', 'Sevilla', 'Tenerife', 'Levante Planas', 'Athletic Club', 'Real Betis', 'Villarreal', 'Alavés', 'Alhama']
actual ranked teams:     ['Barcelona', 'Real Madrid', 'Levante', 'Atlético Madrid', 'Madrid CFF', 'Tenerife', 'Sevilla', 'Real Sociedad', 'Valencia', 'Athletic Club', 'Levante Planas', 'Real Betis', 'Sporting 

C:\Users\Naufal\AppData\Local\Temp\ipykernel_15680\2341830475.py:21: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(mode_at_min_mpd)
C:\Users\Naufal\AppData\Local\Temp\ipykernel_15680\2670775696.py:21: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(mode_at_min_mpd)


New best found: Home Coeff=0.25, Away Coeff=0.25, Margin Coeff=0.00, Date Coeff=0.00, Error=5.8310, Mean%Diff=0.0965
predicted ranked teams:  ['Barcelona', 'Real Madrid', 'Levante', 'Atlético Madrid', 'Madrid CFF', 'Real Sociedad', 'Sevilla', 'Athletic Club', 'Valencia', 'Tenerife', 'Levante Planas', 'Sporting Huelva', 'Real Betis', 'Alavés', 'Alhama', 'Villarreal']
actual ranked teams:     ['Barcelona', 'Real Madrid', 'Levante', 'Atlético Madrid', 'Madrid CFF', 'Tenerife', 'Sevilla', 'Real Sociedad', 'Valencia', 'Athletic Club', 'Levante Planas', 'Real Betis', 'Sporting Huelva', 'Villarreal', 'Alhama', 'Alavés']
--------------------------------------------------------------------------------

Massey Optimization Complete:
Best Home Win Coefficient: 0.25
Best Away Win Coefficient: 0.25
Best Margin Coefficient: 0.00
Best Date Coefficient: 0.00
Best Error: 5.8310
Best Mean Percentage Difference: 0.0965
New best found: Home Coeff=0.25, Away Coeff=0.25, Margin Coeff=0.00, Date Coeff=0.00, 

C:\Users\Naufal\AppData\Local\Temp\ipykernel_15680\2341830475.py:21: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(mode_at_min_mpd)
C:\Users\Naufal\AppData\Local\Temp\ipykernel_15680\2001641750.py:21: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(mode_at_min_mpd)


In [2]:
trueWSL2024_2025 = trueWSL2024_2025 = [
    "Chelsea",        # Champions (1st) — unbeaten season 🏆
    "Arsenal",        # 2nd — UEFA Women’s Champions League spot
    "Manchester Utd",  # 3rd — European qualification
    "Manchester City",    # 4th
    "Brighton",  # 5th
    "Aston Villa",    # 6th
    "Liverpool",      # 7th
    "Everton",        # 8th
    "West Ham",# 9th
    "Leicester City", # 10th
    "Tottenham", # 11th
    "Crystal Palace"
]


# Define the range for home win coefficients
home_win_coeffs = np.arange(0.25, 1.76, 0.25)

# Define the range for away win coefficients
away_win_coeffs = np.arange(0.25, 1.76, 0.25)

# Define the range for away win coefficients
margin_coeffs = np.arange(0.0, 1.01, 0.50)

# Define the range for date  coefficients
date_coeffs = np.arange(0, 1.01, 0.5)

print("Home Win Coefficients:", home_win_coeffs)
print("Away Win Coefficients:", away_win_coeffs)
print("Margin Coefficients:", margin_coeffs)
print("Date Coefficients:", date_coeffs)

train_wsl_2024_2025 = wsl_df[(wsl_df['season'] == '2024-2025') & (wsl_df['gameweek'] <= 11)]

# Grid Search for Optimal Coefficients
best_error = float('inf')
best_mpd = float('inf')
best_home_coeff = None
best_away_coeff = None
best_margin_coeff = None


history = []   # will store one row per try

try_num = 0

for home_coeff in home_win_coeffs:
    for away_coeff in away_win_coeffs:
        for date_coeff in date_coeffs:
            for margin_coeff in margin_coeffs:
                # Run Colley with the current coefficients
                team_ratings, sorted_teams = Colley_weighted_optimized(train_wsl_2024_2025, home_coeff, away_coeff, margin_coeff, date_coeff)

                # Validate the ranking
                err, mpd = validate_ranking(sorted_teams, trueWSL2024_2025)

                try_num += 1
                history.append({
                    "try": try_num,
                    "seasons" : '2024/25',
                    "home_coeff": home_coeff,
                    "away_coeff": away_coeff,
                    "margin_coeff": margin_coeff,
                    "date_coeff": date_coeff,
                    "Error": err,
                    "Mean%Diff": mpd
               })

               # Check if the current combination is better than the best found so far
                if (err < best_error) or (np.isclose(err, best_error) and mpd < best_mpd):
                    best_error = err
                    best_mpd = mpd
                    best_home_coeff = home_coeff
                    best_away_coeff = away_coeff
                    best_margin_coeff = margin_coeff
                    best_date_coeff = date_coeff
                    print(f"New best found: Home Coeff={best_home_coeff:.2f}, Away Coeff={best_away_coeff:.2f}, Margin Coeff={best_margin_coeff:.2f}, Date Coeff={best_date_coeff:.2f}, Error={best_error:.4f}, Mean%Diff={best_mpd:.4f}")
                    print("predicted ranked teams: ", sorted_teams)
                    print("actual ranked teams:    ", trueWSL2024_2025)
                    print("--------------------------------------------------------------------------------")


print("\nColley Optimization Complete:")
print(f"Best Home Win Coefficient: {best_home_coeff:.2f}")
print(f"Best Away Win Coefficient: {best_away_coeff:.2f}")
print(f"Best Margin Coefficient: {best_margin_coeff:.2f}")
print(f"Best Date Coefficient: {best_date_coeff:.2f}")
print(f"Best Error: {best_error:.4f}")
print(f"Best Mean Percentage Difference: {best_mpd:.4f}")



Home Win Coefficients: [0.25 0.5  0.75 1.   1.25 1.5  1.75]
Away Win Coefficients: [0.25 0.5  0.75 1.   1.25 1.5  1.75]
Margin Coefficients: [0.  0.5 1. ]
Date Coefficients: [0.  0.5 1. ]
New best found: Home Coeff=0.25, Away Coeff=0.25, Margin Coeff=0.00, Date Coeff=0.00, Error=5.8310, Mean%Diff=0.0941
predicted ranked teams:  ['Chelsea', 'Arsenal', 'Manchester Utd', 'Manchester City', 'Brighton', 'Tottenham', 'Aston Villa', 'Everton', 'Liverpool', 'Leicester City', 'West Ham', 'Crystal Palace']
actual ranked teams:     ['Chelsea', 'Arsenal', 'Manchester Utd', 'Manchester City', 'Brighton', 'Aston Villa', 'Liverpool', 'Everton', 'West Ham', 'Leicester City', 'Tottenham', 'Crystal Palace']
--------------------------------------------------------------------------------
New best found: Home Coeff=0.25, Away Coeff=0.25, Margin Coeff=0.50, Date Coeff=0.00, Error=5.6569, Mean%Diff=0.0937
predicted ranked teams:  ['Chelsea', 'Arsenal', 'Manchester Utd', 'Manchester City', 'Brighton', 'Totte

In [3]:
train_wsl_2024_2025

,season,gameweek,date,day_of_week,start_time,home_team,away_team,home_score,away_score,score,home_xg,away_xg,venue,attendance,referee,status,result,home_team_url,away_team_url,match_report_url
132,2024-2025,1.0,2024-09-20,Fri,19:00 (11:00),Chelsea,Aston Villa,1.0,0.0,1–0,1.1,0.9,Cherry Red Records Fans' Stadium,4337.0,Kirsty Dowle,completed,home_win,https://fbref.com/en/squads/a6a4e67d/2024-2025...,https://fbref.com/en/squads/53157aa8/2024-2025...,https://fbref.com/en/matches/460a420c/Chelsea-...
133,2024-2025,1.0,2024-09-21,Sat,12:00 (04:00),Manchester Utd,West Ham,3.0,0.0,3–0,1.2,0.9,Old Trafford,8761.0,Cheryl Foster,completed,home_win,https://fbref.com/en/squads/0bbd83f6/2024-2025...,https://fbref.com/en/squads/52d65cea/2024-2025...,https://fbref.com/en/matches/247d25af/Manchest...
134,2024-2025,1.0,2024-09-21,Sat,12:30 (04:30),Brighton,Everton,4.0,0.0,4–0,2.1,0.5,Broadfield Stadium,2125.0,Lauren Impey,completed,home_win,https://fbref.com/en/squads/fa2752bc/2024-2025...,https://fbref.com/en/squads/c4989550/2024-2025...,https://fbref.com/en/matches/86c90fdf/Brighton...
135,2024-2025,1.0,2024-09-22,Sun,12:30 (04:30),Arsenal,Manchester City,2.0,2.0,2–2,2.2,1.4,Emirates Stadium,41818.0,Abigail Byrne,completed,draw,https://fbref.com/en/squads/411b1108/2024-2025...,https://fbref.com/en/squads/9ce68f8a/2024-2025...,https://fbref.com/en/matches/9232742b/Arsenal-...
136,2024-2025,1.0,2024-09-22,Sun,14:00 (06:00),Tottenham,Crystal Palace,4.0,0.0,4–0,4.1,0.3,The BetWright Stadium,1778.0,Amy Fearn,completed,home_win,https://fbref.com/en/squads/e8e4577c/2024-2025...,https://fbref.com/en/squads/78ba1bd4/Crystal-P...,https://fbref.com/en/matches/4d436423/Tottenha...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
193,2024-2025,11.0,2025-01-18,Sat,17:30 (09:30),Everton,Aston Villa,1.0,1.0,1–1,0.4,1.1,Walton Hall Park,973.0,Stacey Fullicks,completed,draw,https://fbref.com/en/squads/c4989550/2024-2025...,https://fbref.com/en/squads/53157aa8/2024-2025...,https://fbref.com/en/matches/9e365585/Everton-...
194,2024-2025,11.0,2025-01-19,Sun,12:30 (04:30),Tottenham,Leicester City,1.0,0.0,1–0,0.0,1.3,The BetWright Stadium,1314.0,Stacey Pearson,completed,home_win,https://fbref.com/en/squads/e8e4577c/2024-2025...,https://fbref.com/en/squads/23bce84e/2024-2025...,https://fbref.com/en/matches/91ce25ab/Tottenha...
195,2024-2025,11.0,2025-01-19,Sun,14:00 (06:00),Arsenal,Crystal Palace,5.0,0.0,5–0,3.2,0.9,Mangata Pay UK Stadium,3576.0,Phoebe Cross,completed,home_win,https://fbref.com/en/squads/411b1108/2024-2025...,https://fbref.com/en/squads/78ba1bd4/Crystal-P...,https://fbref.com/en/matches/0475ac77/Arsenal-...
196,2024-2025,11.0,2025-01-19,Sun,15:00 (07:00),West Ham,Chelsea,0.0,5.0,0–5,0.4,2.3,Chigwell Construction Stadium,2461.0,Amy Fearn,completed,away_win,https://fbref.com/en/squads/52d65cea/2024-2025...,https://fbref.com/en/squads/a6a4e67d/2024-2025...,https://fbref.com/en/matches/10a0825a/West-Ham...


In [4]:
hist_df = pd.DataFrame(history)
hist_df.head(5)

,try,seasons,home_coeff,away_coeff,margin_coeff,date_coeff,Error,Mean%Diff
0,1,2024/25,0.25,0.25,0.0,0.0,5.830952,0.094096
1,2,2024/25,0.25,0.25,0.5,0.0,5.656854,0.093666
2,3,2024/25,0.25,0.25,1.0,0.0,5.477226,0.091682
3,4,2024/25,0.25,0.25,0.0,0.5,5.830952,0.094096
4,5,2024/25,0.25,0.25,0.5,0.5,5.656854,0.093170


In [5]:
from scipy import stats

vars_to_mode = ["home_coeff", "away_coeff", "margin_coeff", "date_coeff"]

def mode_at_min_mpd(df):
    min_mpd = df["Mean%Diff"].min()
    best_rows = df[df["Mean%Diff"] == min_mpd]
    
    mode_values = {}
    for col in vars_to_mode:
        m = best_rows[col].mode()
        # if multiple modes, take the first (deterministic)
        mode_values[col] = float(m.iloc[0])
    
    return pd.Series(mode_values)

# per-season
season_result = (
    hist_df
    .groupby("seasons")
    .apply(mode_at_min_mpd)
)

# ALL seasons
all_result = mode_at_min_mpd(hist_df).to_frame(name="ALL").T

# same orientation as before
final_table = pd.concat([season_result, all_result]).T

final_table

C:\Users\Naufal\AppData\Local\Temp\ipykernel_15680\2508061481.py:21: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(mode_at_min_mpd)


,2024/25,ALL
home_coeff,0.25,0.25
away_coeff,0.75,0.75
margin_coeff,1.00,1.00
date_coeff,1.00,1.00
